# Complete sessions → discrete or continuous space

Loads preprocessed ADS-B session data and the keyed JFK phase sidecar, keeps complete S1–S4 sessions that have JFK takeoff/touchdown phase scope, and exports either spatially discretized tracks or continuous lon/lat/altitude samples with discretized session-relative time. Run `02-Preprocessing/02-tag_flight_phases.py` after preprocessing and before this notebook. Set `MODE` to `"discrete"` or `"continuous"` below.

Discrete mode writes `complete_sessions.json` for [`02-session_space_viz.html`](02-session_space_viz.html); continuous mode writes `continuous_sessions.json` for [`02-session_space_viz_continuous.html`](02-session_space_viz_continuous.html). Both modes use the same complete-session filter and time-bin width.

Completeness uses the preprocessor's unbuffered event counts and `aircraft_session_type`, cross-checked against valid in-sector track points and collection coverage. The JFK phase cohort is selected by event-matched JFK takeoff/touchdown evidence; the point-level phase labels themselves are causal ADS-B estimates and do not use those post-hoc flags. The four completeness categories are:

| kind | preprocessor type | interpretation |
|---|---|---|
| local | S1: takeoff + landing | both in-sector events observed |
| departure | S3: takeoff only | takeoff observed; non-clipped track end represents departure |
| arrival | S2: landing only | non-clipped inbound track start followed by an in-sector landing |
| transit | S4: neither event | non-parked airborne track with no local takeoff/landing |

Arrival/departure/transit are inferred from the existing session labels when direct edge evidence is unavailable. Event counts exclude merge-buffer flags. Dropped sessions include event/type mismatches, collection-window clips, too few valid in-sector points, parked/ground-only tracks, and S3/S4 tracks that leave and re-enter under the same session ID. Edge evidence is retained per session when observed.

The configured sector and edge widths remain those in `01-label_aircraft_id.ipynb`: 35–45°N, 78–68°W, with 0.4° latitude and 0.5° longitude edge bands. This dataset has no samples in those edge bands, so non-local endpoint categories are not direct boundary-crossing measurements. Observed coordinate extrema are never substituted for the configured sector.
QC flags are reported without excluding sessions. For trajectory coordinates and the airborne/parked check, `altitude_phase_ft` is used when available; invalid, out-of-range, and above-ceiling samples are omitted. Exported trajectories are clipped to the configured sector, and time starts at the first retained in-sector sample.

The loader searches for a readable Feather file and skips corrupt/truncated candidates. If needed, set `FEATHER_PATH` below to an explicit preprocessed file.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

HERE = Path.cwd()
VIS_DIR = HERE if (HERE / "complete_sessions.py").exists() else HERE / "03-Visualization"
sys.path.insert(0, str(VIS_DIR))

from complete_sessions import (
    DEFAULT_BINS,
    completeness_counts,
    complete_session_ids,
    export_session_payload,
    load_adsb,
    phase_session_ids,
)

FEATHER_PATH = None  # optionally set a local Path to the preprocessed Feather file

In [ ]:
df, source = load_adsb(FEATHER_PATH)
print("feather:", source)
print("rows:", f"{len(df):,}")
print("labeled sessions:", df["aircraft_session_id"].nunique(dropna=True))
print(df["aircraft_session_type"].value_counts(dropna=False))
print("phase-tagged JFK sessions:", len(phase_session_ids(df, "JFK")))

In [ ]:
MIN_POINTS_PER_SESSION = 2  # minimum valid in-sector coordinates; QC flags are reported separately
keep, summary, box = complete_session_ids(df, min_points=MIN_POINTS_PER_SESSION)
jfk_phase_ids = phase_session_ids(df, "JFK")
phase_summary = summary.loc[summary.index.isin(jfk_phase_ids)]
keep = keep[keep.isin(jfk_phase_ids)]
counts, drops = completeness_counts(phase_summary)
print("configured sector and edge bands:")
display(pd.Series(box))
print("complete kinds:")
display(counts)
print("dropped:")
display(drops)
if "quality_flagged_points" in summary.columns:
    print("QC summary for complete sessions (reported, not excluded):")
    display(summary.loc[keep, ["quality_flagged_points", "quality_flag_events", "n_track_segments"]].sum())
print("complete sessions with JFK phase tags:", f"{len(keep):,}")

In [ ]:
# Show several session paths for one aircraft, including original session labels.
# This reveals whether IDs are split across continuous tracks or merged across long gaps.
aircraft_counts = df.groupby("aircraft")["aircraft_session_id"].nunique(dropna=True)
sample_aircraft = aircraft_counts[aircraft_counts >= 2].sort_values(ascending=False).index[0]
ac = df.loc[df["aircraft"] == sample_aircraft].sort_values("ts")
fig, ax = plt.subplots(figsize=(10, 7))
for sid, tr in ac.groupby("aircraft_session_id", sort=False, dropna=False):
    label = str(sid) if pd.notna(sid) else "unlabeled"
    ax.plot(tr["lon"], tr["lat"], marker=".", ms=2, lw=1, label=label)
    mid = tr.iloc[len(tr) // 2]
    ax.annotate(label, (mid["lon"], mid["lat"]), fontsize=7)
ax.scatter(ac.groupby("aircraft_session_id", dropna=False).first()["lon"], ac.groupby("aircraft_session_id", dropna=False).first()["lat"], marker="o", s=22, facecolors="none", edgecolors="white", label="session start")
ax.set_xlabel("lon (x)")
ax.set_ylabel("lat (y)")
ax.set_aspect("equal", adjustable="box")
ax.legend(fontsize=7, ncol=2)
ax.set_title(f"Aircraft {sample_aircraft}: trajectories by aircraft_session_id")
plt.show()

# Compare preprocessor id (gap/event based) with aircraft_session_id.
display(ac.groupby("aircraft_session_id", dropna=False).agg(
    rows=("ts", "size"), id_count=("id", "nunique"),
    t_start=("ts", "min"), t_end=("ts", "max"),
    takeoffs=("takeoff", "sum"), touchdowns=("touchdown", "sum"),
).reset_index())

# Inspect most frequent session assignments and collection-level event counters.
display(phase_summary.groupby(["session_type", "complete_kind", "drop_reason"], dropna=False).size().rename("sessions").reset_index())
print("Rows in configured edge bands:", int((df["lon"].lt(box["lon_min"] + box["lon_range"]) | df["lon"].gt(box["lon_max"] - box["lon_range"]) | df["lat"].lt(box["lat_min"] + box["lat_range"]) | df["lat"].gt(box["lat_max"] - box["lat_range"])).sum()))

In [ ]:
MODE = "discrete"  # choose "continuous" to retain raw lon/lat/altitude samples
BINS = dict(DEFAULT_BINS)  # dx deg, dy deg, dz ft, dt s; continuous mode uses dt only
if MODE not in {"discrete", "continuous"}:
    raise ValueError("MODE must be 'discrete' or 'continuous'")
print("export mode:", MODE)
print("time-step seconds:", BINS["dt"])

In [ ]:
json_name = "complete_sessions.json" if MODE == "discrete" else "continuous_sessions.json"
out, meta = export_session_payload(
    df=df,
    source=source,
    session_ids=keep,
    summary=summary,
    box=box,
    mode=MODE,
    phase_airport="JFK",
    bins=BINS,
    out_path=VIS_DIR / json_name,
)
print(meta)
print("wrote", out.resolve())

Open the player (fetch of the JSON fails on `file://`; serve the folder or use **Load JSON**):

```bash
cd 03-Visualization
python -m http.server 8765
```

For MODE = 'discrete', open http://127.0.0.1:8765/02-session_space_viz.html. For MODE = 'continuous', open http://127.0.0.1:8765/02-session_space_viz_continuous.html.

The discrete player visualizes binned (x, y, z, t) occupancy and supports voxel re-zeroing. The continuous player retains continuous lon/lat/altitude samples while using discretized time. Both pages provide 2D and orbitable 3D views.